# Bybit — Open Interest: Extraction and 1h Panel

This notebook builds the open-interest (OI) dataset used in the empirical analysis. It retrieves OI observations from Bybit’s REST API, standardizes timestamps in UTC, and outputs an hourly panel designed to be merged on `ts` with the OHLCV (klines) and funding datasets.

## Context
Open interest is a key state variable in derivatives markets and is used downstream together with prices/returns and funding. The rest of the empirical pipeline is organized on an **hourly time grid**, so this notebook produces an OI series indexed at 1-hour frequency for consistent joins.

## Source and time conventions
- **Provider:** Bybit
- **Interface:** REST API (open interest endpoint)
- **Time standard:** all timestamps are handled in **UTC**
- **Window convention:** extraction covers **[START, END_EXCL)** (end bound is *exclusive*)

## Inputs (set in the configuration cell)
- `SYMBOL` (e.g., `ETHUSDT`)
- `CATEGORY` (market type used in the study design, e.g., `linear`)
- `START` and `END_EXCL` (UTC; `END_EXCL` is exclusive)

## What the notebook does (in order)
1) **Setup**: reads parameters, defines the time window, and prepares output paths under `data/`.
2) **Retrieval**: queries the API over the requested window (paged/segmented requests, with basic retry logic).
3) **Normalization**: converts raw payloads into a tidy table (typed columns, UTC datetimes, sorted and de-duplicated).
4) **Hourly panel construction**: produces a 1-hour indexed series consistent with the project’s merge key (`ts`, UTC).
5) **QA and export**: writes raw, processed, QA, and a manifest with run parameters and file hashes.

## Outputs
Files are written under `data/`:

- `data/raw/...jsonl` — raw API payloads (audit trail)
- `data/processed/...parquet` — hourly OI panel (analysis-ready)
- `data/qa/..._qa.json` — QA summary (gaps, duplicates, bounds, missingness)
- `data/manifests/..._manifest.json` — parameters and SHA256 hashes of the outputs

## Hourly panel (method note)
Open interest may be returned at discrete timestamps depending on the endpoint and market. This notebook maps observations onto an hourly index used throughout the empirical pipeline. The alignment rule (e.g., selecting the last observation within each hour, or carrying forward the most recent value) should remain consistent across datasets so merges on `ts` are well-defined.

## Processed dataset (schema)
One row per hour (UTC):
- `ts` (datetime UTC; hourly index)
- `open_interest` (float)
- `symbol`, `category`
- `source` (e.g., `"bybit"`)
- `ingested_at_utc` (datetime UTC)

Additional columns may be present if returned by the API (kept as metadata rather than inputs to the merge key).

## Reproducibility note
The pipeline is reproducible given the same code and parameters. Exact bit-for-bit reproducibility depends on the provider returning identical historical data for the same query window. The manifest hashes record the exact artifacts produced for each run.

## Quick notes
- If something looks off, check the QA JSON first (time bounds, duplicates, missing hours) and confirm the configured window and symbol.
- The extraction window is **[START, END_EXCL)**; adjacent runs can be concatenated without overlap.
- For downstream merges, join on `ts` (UTC) and keep a single alignment rule for all hourly series.

In [1]:
import json, time, hashlib, random
from pathlib import Path
from typing import Dict, Any, List, Tuple

import pandas as pd
import numpy as np
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ===== CONFIG =====
OUT_DIR = Path("raw_data_lake") / "bybit_open_interest_1h"
BASE_URL = "https://api.bybit.com"

CATEGORY = "linear"
SYMBOL = "ETHUSDT"
INTERVAL_TIME = "1h"      # allowed: 5min, 15min, 30min, 1h, 4h, 1d :contentReference[oaicite:1]{index=1}
LIMIT = 200              # max 200 :contentReference[oaicite:2]{index=2}

START = pd.Timestamp("2021-03-15T00:00:00Z")
END_EXCL = pd.Timestamp("2025-12-01T00:00:00Z")

# HTTP
MIN_REQUEST_INTERVAL_S = 0.6
TIMEOUT = (10, 120)
RETRIES_TOTAL = 8
BACKOFF_FACTOR = 1.0

# pagination
CHUNK_DAYS = 365
MAX_PAGES_PER_CHUNK = 20000

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR =", OUT_DIR)
print("Range:", START, "->", END_EXCL, "(end exclusive)")


OUT_DIR = raw_data_lake/bybit_open_interest_1h
Range: 2021-03-15 00:00:00+00:00 -> 2025-12-01 00:00:00+00:00 (end exclusive)


In [3]:
def ms(ts: pd.Timestamp) -> int:
    return int(ts.value // 10**6)

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def append_jsonl(path: Path, obj: Dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def utc_hour_grid(start: pd.Timestamp, end_excl: pd.Timestamp) -> pd.DatetimeIndex:
    return pd.date_range(start=start, end=end_excl, freq="h", inclusive="left", tz="UTC")

def now_utc_iso() -> str:
    return pd.Timestamp.utcnow().isoformat()


In [5]:
def make_session() -> requests.Session:
    s = requests.Session()
    retry = Retry(
        total=RETRIES_TOTAL,
        backoff_factor=BACKOFF_FACTOR,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
        raise_on_status=False,
    )
    s.mount("https://", HTTPAdapter(max_retries=retry))
    return s

SESSION = make_session()
_last_req_t = 0.0

def _pace():
    global _last_req_t
    now = time.monotonic()
    wait = MIN_REQUEST_INTERVAL_S - (now - _last_req_t)
    if wait > 0:
        time.sleep(wait + random.uniform(0.0, 0.15))
    _last_req_t = time.monotonic()

def bybit_get(path: str, params: Dict[str, Any]) -> Dict[str, Any]:
    url = f"{BASE_URL}{path}"

    for attempt in range(1, RETRIES_TOTAL + 1):
        _pace()
        t0 = time.time()
        r = SESSION.get(url, params=params, timeout=TIMEOUT)
        elapsed = time.time() - t0

        try:
            data = r.json()
        except Exception:
            data = {"retCode": None, "retMsg": "Non-JSON response", "_http_status": r.status_code}

        if data.get("retCode") == 10006:
            sleep_s = min(60.0, (2 ** (attempt - 1)) * 0.8 + random.uniform(0.0, 0.8))
            print(f"[rate_limit] 10006 attempt={attempt}/{RETRIES_TOTAL} sleep={sleep_s:.2f}s")
            time.sleep(sleep_s)
            continue

        return {
            "response": data,
            "http_status": r.status_code,
            "elapsed_s": elapsed,
            "url": url,
        }

    raise RuntimeError("Rate limit persists after retries.")


In [7]:
def fetch_open_interest_raw(start: pd.Timestamp, end_excl: pd.Timestamp) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    pages_path = OUT_DIR / "pages.jsonl"
    if pages_path.exists():
        pages_path.unlink()

    all_items: List[dict] = []
    requests_made = stalls = empty_pages = 0

    chunk_start = start
    while chunk_start < end_excl:
        chunk_end = min(chunk_start + pd.Timedelta(days=CHUNK_DAYS), end_excl)
        print(f"[oi] chunk {chunk_start:%Y-%m-%d} -> {chunk_end:%Y-%m-%d}")

        cursor_end = chunk_end
        prev_sig = None
        safety = 0

        while True:
            safety += 1
            if safety > MAX_PAGES_PER_CHUNK:
                raise RuntimeError("Too many pages in chunk; reduce CHUNK_DAYS.")

            params = {
                "category": CATEGORY,
                "symbol": SYMBOL,
                "intervalTime": INTERVAL_TIME,
                "startTime": ms(chunk_start),
                "endTime": ms(cursor_end),
                "limit": LIMIT,
            }

            page = bybit_get("/v5/market/open-interest", params)
            res = page["response"]
            requests_made += 1

            append_jsonl(pages_path, {
                "t": time.time(),
                "path": "/v5/market/open-interest",
                "params": params,
                "http_status": page["http_status"],
                "elapsed_s": page["elapsed_s"],
                "retCode": res.get("retCode"),
                "retMsg": res.get("retMsg"),
            })

            if res.get("retCode") != 0:
                raise RuntimeError(f"Bybit retCode={res.get('retCode')} retMsg={res.get('retMsg')}")

            lst = (res.get("result") or {}).get("list") or []
            if not lst:
                empty_pages += 1
                break

            # Bybit OI list items contain "timestamp" (ms)
            if "timestamp" not in lst[0]:
                raise ValueError(f"Cannot find 'timestamp' in OI payload keys={list(lst[0].keys())}")

            ts = [int(x["timestamp"]) for x in lst]
            page_min_ms, page_max_ms = min(ts), max(ts)
            sig = (len(lst), page_min_ms, page_max_ms)
            if prev_sig is not None and sig == prev_sig:
                stalls += 1
                print("[oi][warn] repeated page signature -> break chunk")
                break
            prev_sig = sig

            all_items.extend(lst)

            page_min = pd.to_datetime(page_min_ms, unit="ms", utc=True)
            if page_min <= chunk_start:
                break

            new_cursor_end = page_min - pd.Timedelta(milliseconds=1)
            if new_cursor_end >= cursor_end:
                stalls += 1
                print("[oi][warn] cursor not moving -> break chunk")
                break
            cursor_end = new_cursor_end

        chunk_start = chunk_end

    df = pd.DataFrame(all_items)

    summary = {
        "requests_made": requests_made,
        "rows_raw": int(len(all_items)),
        "stalls": stalls,
        "empty_pages": empty_pages,
    }
    return df, summary


In [9]:
def normalize_open_interest_1h(df_raw: pd.DataFrame, start: pd.Timestamp, end_excl: pd.Timestamp):
    idx = utc_hour_grid(start, end_excl)

    if df_raw.empty:
        out = pd.DataFrame({"date": idx, "open_interest_eth": np.nan})
        qa = {"expected_hours": int(len(idx)), "rows_after_dedup": 0, "missing_oi": int(out["open_interest_eth"].isna().sum())}
        return out, qa

    df = df_raw.copy()

    # Required fields per Bybit doc examples: timestamp, openInterest :contentReference[oaicite:4]{index=4}
    if "timestamp" not in df.columns or "openInterest" not in df.columns:
        raise ValueError(f"OI columns missing. Have={list(df.columns)}")

    df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True).dt.floor("h")
    df["open_interest_eth"] = pd.to_numeric(df["openInterest"], errors="coerce")

    df = df.sort_values("date").drop_duplicates("date", keep="last")
    df = df[(df["date"] >= start) & (df["date"] < end_excl)].copy()

    out = pd.DataFrame({"date": idx}).merge(df[["date","open_interest_eth"]], on="date", how="left")

    qa = {
        "expected_hours": int(len(idx)),
        "rows_after_dedup": int(len(df)),
        "missing_oi": int(out["open_interest_eth"].isna().sum()),
        "min": float(out["open_interest_eth"].min(skipna=True)) if out["open_interest_eth"].notna().any() else None,
        "max": float(out["open_interest_eth"].max(skipna=True)) if out["open_interest_eth"].notna().any() else None,
    }
    return out, qa


In [11]:
df_raw, http_summary = fetch_open_interest_raw(START, END_EXCL)
df_norm, qa = normalize_open_interest_1h(df_raw, START, END_EXCL)

raw_csv = OUT_DIR / "open_interest_raw.csv"
raw_parq = OUT_DIR / "open_interest_raw.parquet"
norm_parq = OUT_DIR / "open_interest_1h.parquet"
qa_path = OUT_DIR / "qa_report.json"

df_raw.to_csv(raw_csv, index=False)
df_raw.to_parquet(raw_parq, index=False, engine="pyarrow")

df_norm.to_parquet(norm_parq, index=False, engine="pyarrow")
qa_path.write_text(json.dumps(qa, indent=2), encoding="utf-8")

manifest = {
    "dataset": "bybit_open_interest_1h",
    "collected_at_utc": now_utc_iso(),
    "config": {
        "base_url": BASE_URL,
        "category": CATEGORY,
        "symbol": SYMBOL,
        "intervalTime": INTERVAL_TIME,
        "limit": LIMIT,
        "start": str(START),
        "end_excl": str(END_EXCL),
        "chunk_days": CHUNK_DAYS,
        "min_request_interval_s": MIN_REQUEST_INTERVAL_S,
        "timeout": TIMEOUT,
        "retries_total": RETRIES_TOTAL,
        "backoff_factor": BACKOFF_FACTOR,
    },
    "files": {
        "raw_csv": str(raw_csv),
        "raw_parquet": str(raw_parq),
        "norm_parquet": str(norm_parq),
        "pages_jsonl": str(OUT_DIR / "pages.jsonl"),
        "qa_report": str(qa_path),
    },
    "hashes": {
        "raw_csv_sha256": sha256_file(raw_csv),
        "raw_parquet_sha256": sha256_file(raw_parq),
        "norm_parquet_sha256": sha256_file(norm_parq),
        "qa_report_sha256": sha256_file(qa_path),
    },
    "http_summary": http_summary,
    "qa": qa,
}

(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("✅ Saved:", norm_parq)
print("HTTP summary:", http_summary)
print("QA:", qa)

df_norm.head()

[oi] chunk 2021-03-15 -> 2022-03-15
[oi] chunk 2022-03-15 -> 2023-03-15
[oi] chunk 2023-03-15 -> 2024-03-14
[oi] chunk 2024-03-14 -> 2025-03-14
[oi] chunk 2025-03-14 -> 2025-12-01
✅ Saved: raw_data_lake/bybit_open_interest_1h/open_interest_1h.parquet
HTTP summary: {'requests_made': 208, 'rows_raw': 41333, 'stalls': 0, 'empty_pages': 0}
QA: {'expected_hours': 41328, 'rows_after_dedup': 41328, 'missing_oi': 0, 'min': 23189.6, 'max': 1263195.04}


,date,open_interest_eth
0,2021-03-15 00:00:00+00:00,67850.55
1,2021-03-15 01:00:00+00:00,66316.54
2,2021-03-15 02:00:00+00:00,66360.84
3,2021-03-15 03:00:00+00:00,67136.36
4,2021-03-15 04:00:00+00:00,67051.12
